# ModeSwitch-LLM: Hivansh Selector Notebook
**Author:** Hivansh Dhakne (hd2296)  
**Purpose:** Repo-aligned notebook for Hivansh's project responsibilities

This notebook is written specifically for the current `ModeSwitch-LLM` repo after pulling in
Aman's newer benchmark additions. It is meant to satisfy Hivansh's role in the proposal:

- design the phase-aware selector
- implement the phase-agnostic comparison baseline
- compare against static baselines
- use runtime signals such as prompt length, predicted output length, memory pressure, and latency targets
- stay ready to plug in real benchmark outputs from Aman/Ali


## Aman Additions Scanned

I scanned the new Aman-facing additions that now matter for the selector:

- `baseline_smoke_test (3) (1).ipynb`
  This shows controller-style benchmarking across the repo mode set, including
  `fp16_baseline`, `int8_quant`, `awq_4bit`, `gptq_4bit`, `speculative_decoding`,
  `kv_cache_compression`, `prefix_caching`, `chunked_prefill`, `continuous_batching`,
  `cuda_graphs`, and a hybrid mode.
- `stress_test (2).ipynb`
  This adds long-context stress cases like `ctx3840_out256` and `ctx4032_out64`, which matter
  directly for selector logic because they expose different prefill/decode bottlenecks.

Selector implications from those additions:

- long prompts and shared-prefix workloads make prefill-sensitive modes important
- long outputs make decode-sensitive modes more important
- memory pressure and long context make KV-cache and quantized tradeoffs more relevant
- the selector should use the repo's actual mode names and benchmark shapes, not placeholder names


In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Tuple

import numpy as np
import pandas as pd


class SelectorMode(str, Enum):
    FP16_BASELINE = "fp16_baseline"
    INT8_QUANT = "int8_quant"
    AWQ_4BIT = "awq_4bit"
    GPTQ_4BIT = "gptq_4bit"
    SPECULATIVE_DECODING = "speculative_decoding"
    KV_CACHE_COMPRESSION = "kv_cache_compression"
    PREFIX_CACHING = "prefix_caching"
    CHUNKED_PREFILL = "chunked_prefill"
    CONTINUOUS_BATCHING = "continuous_batching"
    CUDA_GRAPHS = "cuda_graphs"


@dataclass(frozen=True)
class WorkloadKey:
    prompt_bucket: str
    output_bucket: str
    memory_bucket: str


@dataclass(frozen=True)
class RequestSignals:
    prompt_length: int
    predicted_output_length: int
    memory_pressure: float
    ttft_target_ms: float
    tbt_target_ms: float


@dataclass
class ModeProfile:
    mode: SelectorMode
    phase: str
    phase_latency_ms: float
    throughput_tok_s: float
    memory_mb: float
    energy_per_token_j: float
    quality_score: float
    switch_overhead_ms: float = 0.0
    notes: str = ""


@dataclass
class SelectionDecision:
    selector_name: str
    prefill_mode: SelectorMode
    decode_mode: SelectorMode
    switch_overhead_ms: float = 0.0


@dataclass
class BenchmarkDB:
    profiles: Dict[Tuple[SelectorMode, str, WorkloadKey], ModeProfile] = field(default_factory=dict)

    def add(self, mode: SelectorMode, phase: str, workload_key: WorkloadKey, profile: ModeProfile) -> None:
        self.profiles[(mode, phase, workload_key)] = profile

    def get(self, mode: SelectorMode, phase: str, workload_key: WorkloadKey) -> Optional[ModeProfile]:
        return self.profiles.get((mode, phase, workload_key))

    def available_modes(self, phase: str, workload_key: WorkloadKey) -> List[SelectorMode]:
        return [mode for mode in SelectorMode if self.get(mode, phase, workload_key) is not None]

    def to_dataframe(self) -> pd.DataFrame:
        rows = []
        for (mode, phase, key), profile in self.profiles.items():
            rows.append(
                {
                    "mode_name": mode.value,
                    "phase": phase,
                    "prompt_bucket": key.prompt_bucket,
                    "output_bucket": key.output_bucket,
                    "memory_bucket": key.memory_bucket,
                    "phase_latency_ms": profile.phase_latency_ms,
                    "throughput_tok_s": profile.throughput_tok_s,
                    "memory_mb": profile.memory_mb,
                    "energy_per_token_j": profile.energy_per_token_j,
                    "quality_score": profile.quality_score,
                    "switch_overhead_ms": profile.switch_overhead_ms,
                }
            )
        return pd.DataFrame(rows)

    @classmethod
    def from_result_rows(cls, rows: Iterable[Mapping[str, object]]) -> "BenchmarkDB":
        db = cls()
        for row in rows:
            mode = _resolve_mode_name(row.get("mode_name") or row.get("inference_mode"))
            if mode is None:
                continue
            key = _workload_key_from_row(row)
            if key is None:
                continue

            prefill_latency = _first_number(row, "ttft_ms_mean", "ttft_ms")
            decode_latency = _first_number(row, "tbt_mean_ms", "avg_tbt_ms_mean", "avg_tbt_ms")
            throughput = _first_number(
                row,
                "decode_throughput_tps",
                "tokens_per_second_mean",
                "tokens_per_second",
            ) or 0.0
            memory_mb = _first_number(row, "peak_gpu_memory_mb_mean", "peak_gpu_memory_mb")
            if memory_mb is None:
                peak_vram_gb = _first_number(row, "peak_vram_gb")
                if peak_vram_gb is not None:
                    memory_mb = peak_vram_gb * 1024.0
            memory_mb = memory_mb or 0.0
            energy_per_token_j = _first_number(row, "energy_per_token_j_mean", "energy_per_token_j") or 0.0
            quality_score = _first_number(
                row,
                "baseline_similarity_rouge_l_f1_mean",
                "reference_rouge_l_f1_mean",
                "quality_score",
            ) or 1.0

            if prefill_latency is not None:
                db.add(
                    mode,
                    "prefill",
                    key,
                    ModeProfile(
                        mode=mode,
                        phase="prefill",
                        phase_latency_ms=float(prefill_latency),
                        throughput_tok_s=float(throughput),
                        memory_mb=float(memory_mb),
                        energy_per_token_j=float(energy_per_token_j),
                        quality_score=float(quality_score),
                        switch_overhead_ms=_default_switch_overhead_ms(mode),
                    ),
                )
            if decode_latency is not None:
                db.add(
                    mode,
                    "decode",
                    key,
                    ModeProfile(
                        mode=mode,
                        phase="decode",
                        phase_latency_ms=float(decode_latency),
                        throughput_tok_s=float(throughput),
                        memory_mb=float(memory_mb),
                        energy_per_token_j=float(energy_per_token_j),
                        quality_score=float(quality_score),
                        switch_overhead_ms=_default_switch_overhead_ms(mode),
                    ),
                )
        return db


def prompt_bucket(prompt_length: int) -> str:
    if prompt_length < 256:
        return "short_prompt"
    if prompt_length < 1024:
        return "medium_prompt"
    return "long_prompt"


def output_bucket(predicted_output_length: int) -> str:
    if predicted_output_length <= 64:
        return "short_output"
    if predicted_output_length <= 256:
        return "medium_output"
    return "long_output"


def memory_bucket(memory_pressure: float) -> str:
    if memory_pressure < 0.6:
        return "baseline"
    if memory_pressure < 0.8:
        return "mem_pressure_50"
    return "mem_pressure_75"


def workload_key_from_signals(signals: RequestSignals) -> WorkloadKey:
    return WorkloadKey(
        prompt_bucket=prompt_bucket(signals.prompt_length),
        output_bucket=output_bucket(signals.predicted_output_length),
        memory_bucket=memory_bucket(signals.memory_pressure),
    )


def _default_switch_overhead_ms(mode: SelectorMode) -> float:
    if mode == SelectorMode.SPECULATIVE_DECODING:
        return 14.0
    if mode == SelectorMode.KV_CACHE_COMPRESSION:
        return 12.0
    if mode in {
        SelectorMode.PREFIX_CACHING,
        SelectorMode.CHUNKED_PREFILL,
        SelectorMode.CONTINUOUS_BATCHING,
        SelectorMode.CUDA_GRAPHS,
    }:
        return 8.0
    return 10.0


def build_synthetic_selector_db(seed: int = 42) -> BenchmarkDB:
    rng = np.random.default_rng(seed)
    db = BenchmarkDB()

    base = {
        SelectorMode.FP16_BASELINE: (105.0, 22.0, 32800.0, 0.0047, 1.000),
        SelectorMode.INT8_QUANT: (48.0, 14.0, 31000.0, 0.0022, 0.986),
        SelectorMode.AWQ_4BIT: (56.0, 13.0, 28700.0, 0.0020, 0.968),
        SelectorMode.GPTQ_4BIT: (52.0, 11.5, 28100.0, 0.0019, 0.962),
        SelectorMode.SPECULATIVE_DECODING: (72.0, 8.5, 33500.0, 0.0025, 0.992),
        SelectorMode.KV_CACHE_COMPRESSION: (61.0, 16.0, 25000.0, 0.0023, 0.978),
        SelectorMode.PREFIX_CACHING: (68.0, 21.0, 32000.0, 0.0041, 1.000),
        SelectorMode.CHUNKED_PREFILL: (66.0, 20.0, 31200.0, 0.0039, 1.000),
        SelectorMode.CONTINUOUS_BATCHING: (82.0, 10.5, 33000.0, 0.0026, 0.955),
        SelectorMode.CUDA_GRAPHS: (58.0, 18.0, 32100.0, 0.0038, 1.000),
    }

    prompt_scale = {"short_prompt": 0.35, "medium_prompt": 1.0, "long_prompt": 2.1}
    output_scale = {"short_output": 0.55, "medium_output": 1.0, "long_output": 1.8}
    memory_scale = {"baseline": 1.0, "mem_pressure_50": 1.08, "mem_pressure_75": 1.18}

    for mode, (prefill_ms, decode_ms, memory_mb, energy_j, quality) in base.items():
        for p_bucket, p_scale in prompt_scale.items():
            for o_bucket, o_scale in output_scale.items():
                for m_bucket, m_scale in memory_scale.items():
                    ttft = prefill_ms * p_scale
                    tbt = decode_ms * (0.85 + 0.15 * o_scale + 0.10 * max(p_scale - 1.0, 0.0))
                    mem = memory_mb * (0.90 + 0.10 * p_scale + 0.08 * o_scale) * m_scale
                    energy = energy_j * (0.95 + 0.08 * p_scale + 0.10 * o_scale)
                    q = quality

                    if mode == SelectorMode.PREFIX_CACHING:
                        if p_bucket == "long_prompt":
                            ttft *= 0.55
                            energy *= 0.78
                        elif p_bucket == "short_prompt":
                            ttft *= 1.10
                    if mode == SelectorMode.CHUNKED_PREFILL:
                        if p_bucket == "long_prompt":
                            ttft *= 0.62
                        elif p_bucket == "short_prompt":
                            ttft *= 1.08
                    if mode == SelectorMode.SPECULATIVE_DECODING:
                        if o_bucket == "long_output":
                            tbt *= 0.65
                            energy *= 0.85
                        elif o_bucket == "short_output":
                            tbt *= 1.05
                    if mode == SelectorMode.KV_CACHE_COMPRESSION:
                        if o_bucket != "short_output":
                            tbt *= 0.82
                        if m_bucket != "baseline":
                            tbt *= 0.85
                            mem *= 0.72
                            energy *= 0.92
                    if mode == SelectorMode.CONTINUOUS_BATCHING:
                        if o_bucket == "long_output":
                            tbt *= 0.74
                        else:
                            tbt *= 1.04
                    if mode == SelectorMode.CUDA_GRAPHS:
                        if p_bucket != "long_prompt":
                            ttft *= 0.72
                        else:
                            ttft *= 0.90

                    ttft *= rng.normal(1.0, 0.03)
                    tbt *= rng.normal(1.0, 0.03)
                    mem *= rng.normal(1.0, 0.02)
                    energy *= rng.normal(1.0, 0.02)
                    q = max(0.90, min(1.0, q))

                    key = WorkloadKey(p_bucket, o_bucket, m_bucket)
                    db.add(
                        mode,
                        "prefill",
                        key,
                        ModeProfile(
                            mode=mode,
                            phase="prefill",
                            phase_latency_ms=float(ttft),
                            throughput_tok_s=float(1000.0 / max(ttft, 1e-6) * 128.0),
                            memory_mb=float(mem),
                            energy_per_token_j=float(energy),
                            quality_score=float(q),
                            switch_overhead_ms=_default_switch_overhead_ms(mode),
                        ),
                    )
                    db.add(
                        mode,
                        "decode",
                        key,
                        ModeProfile(
                            mode=mode,
                            phase="decode",
                            phase_latency_ms=float(tbt),
                            throughput_tok_s=float(1000.0 / max(tbt, 1e-6)),
                            memory_mb=float(mem),
                            energy_per_token_j=float(energy),
                            quality_score=float(q),
                            switch_overhead_ms=_default_switch_overhead_ms(mode),
                        ),
                    )
    return db


class BaseSelector(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @abstractmethod
    def select_request(self, signals: RequestSignals, db: BenchmarkDB) -> SelectionDecision:
        raise NotImplementedError


class StaticSelector(BaseSelector):
    def __init__(self, mode: SelectorMode):
        self.mode = mode

    @property
    def name(self) -> str:
        return f"static_{self.mode.value}"

    def select_request(self, signals: RequestSignals, db: BenchmarkDB) -> SelectionDecision:
        return SelectionDecision(self.name, self.mode, self.mode, 0.0)


class PhaseAgnosticSelector(BaseSelector):
    def __init__(
        self,
        w_energy: float = 1.0,
        w_latency: float = 1.0,
        w_quality: float = 3.0,
        w_memory: float = 0.6,
        quality_floor: float = 0.95,
    ):
        self.w_energy = w_energy
        self.w_latency = w_latency
        self.w_quality = w_quality
        self.w_memory = w_memory
        self.quality_floor = quality_floor

    @property
    def name(self) -> str:
        return "phase_agnostic"

    def _request_cost(self, mode: SelectorMode, signals: RequestSignals, db: BenchmarkDB) -> float:
        key = workload_key_from_signals(signals)
        prefill = db.get(mode, "prefill", key)
        decode = db.get(mode, "decode", key)
        if prefill is None or decode is None:
            return float("inf")
        total_latency_ms = prefill.phase_latency_ms + signals.predicted_output_length * decode.phase_latency_ms
        total_energy_j = (
            signals.prompt_length * prefill.energy_per_token_j
            + signals.predicted_output_length * decode.energy_per_token_j
        )
        quality_score = min(prefill.quality_score, decode.quality_score)
        memory_mb = max(prefill.memory_mb, decode.memory_mb)
        cost = (
            self.w_energy * total_energy_j
            + self.w_latency * (total_latency_ms / 1000.0)
            + self.w_memory * (signals.memory_pressure * memory_mb / 10000.0)
            - self.w_quality * quality_score
        )
        if prefill.phase_latency_ms > signals.ttft_target_ms:
            cost += 4.0 * ((prefill.phase_latency_ms - signals.ttft_target_ms) / signals.ttft_target_ms)
        if decode.phase_latency_ms > signals.tbt_target_ms:
            cost += 4.0 * ((decode.phase_latency_ms - signals.tbt_target_ms) / signals.tbt_target_ms)
        if quality_score < self.quality_floor:
            cost += 5.0 * (self.quality_floor - quality_score)
        return cost

    def select_request(self, signals: RequestSignals, db: BenchmarkDB) -> SelectionDecision:
        best_mode = SelectorMode.FP16_BASELINE
        best_cost = float("inf")
        for mode in SelectorMode:
            cost = self._request_cost(mode, signals, db)
            if cost < best_cost:
                best_cost = cost
                best_mode = mode
        return SelectionDecision(self.name, best_mode, best_mode, 0.0)


class PhaseAwareRuleSelector(BaseSelector):
    def __init__(
        self,
        w_energy: float = 1.0,
        w_latency: float = 1.1,
        w_quality: float = 3.0,
        w_memory: float = 0.7,
        quality_floor: float = 0.95,
        allow_switching: bool = True,
        switch_penalty_ms: float = 12.0,
    ):
        self.w_energy = w_energy
        self.w_latency = w_latency
        self.w_quality = w_quality
        self.w_memory = w_memory
        self.quality_floor = quality_floor
        self.allow_switching = allow_switching
        self.switch_penalty_ms = switch_penalty_ms

    @property
    def name(self) -> str:
        return "phase_aware_rule_switch" if self.allow_switching else "phase_aware_rule_single_mode"

    def _phase_cost(self, mode: SelectorMode, phase: str, signals: RequestSignals, db: BenchmarkDB) -> float:
        key = workload_key_from_signals(signals)
        profile = db.get(mode, phase, key)
        if profile is None:
            return float("inf")
        tokens = signals.prompt_length if phase == "prefill" else signals.predicted_output_length
        target_ms = signals.ttft_target_ms if phase == "prefill" else signals.tbt_target_ms
        latency_ms = profile.phase_latency_ms
        cost = (
            self.w_energy * profile.energy_per_token_j * tokens
            + self.w_latency * (latency_ms / max(target_ms, 1.0))
            + self.w_memory * (signals.memory_pressure * profile.memory_mb / 10000.0)
            - self.w_quality * profile.quality_score
        )
        if profile.quality_score < self.quality_floor:
            cost += 6.0 * (self.quality_floor - profile.quality_score)
        if latency_ms > target_ms:
            cost += 5.0 * ((latency_ms - target_ms) / target_ms)
        if phase == "decode" and signals.predicted_output_length >= 256:
            cost += 0.20 * signals.predicted_output_length * (latency_ms / 1000.0)
        if phase == "prefill" and signals.prompt_length >= 1024:
            cost += 0.15 * signals.prompt_length * (latency_ms / 10000.0)
        return cost

    def _best_phase_mode(self, phase: str, signals: RequestSignals, db: BenchmarkDB) -> SelectorMode:
        key = workload_key_from_signals(signals)
        candidates = db.available_modes(phase, key)
        best_mode = SelectorMode.FP16_BASELINE
        best_cost = float("inf")
        for mode in candidates:
            cost = self._phase_cost(mode, phase, signals, db)
            if cost < best_cost:
                best_cost = cost
                best_mode = mode
        return best_mode

    def select_request(self, signals: RequestSignals, db: BenchmarkDB) -> SelectionDecision:
        if not self.allow_switching:
            agnostic = PhaseAgnosticSelector(
                w_energy=self.w_energy,
                w_latency=self.w_latency,
                w_quality=self.w_quality,
                w_memory=self.w_memory,
                quality_floor=self.quality_floor,
            )
            base = agnostic.select_request(signals, db)
            return SelectionDecision(self.name, base.prefill_mode, base.decode_mode, 0.0)

        prefill_mode = self._best_phase_mode("prefill", signals, db)
        decode_mode = self._best_phase_mode("decode", signals, db)
        switch_overhead_ms = 0.0
        if prefill_mode != decode_mode:
            switch_overhead_ms = max(
                self.switch_penalty_ms,
                _default_switch_overhead_ms(prefill_mode),
                _default_switch_overhead_ms(decode_mode),
            )
            same_mode_cost = self._phase_cost(prefill_mode, "prefill", signals, db) + self._phase_cost(prefill_mode, "decode", signals, db)
            switched_cost = self._phase_cost(prefill_mode, "prefill", signals, db) + self._phase_cost(decode_mode, "decode", signals, db) + switch_overhead_ms / max(signals.ttft_target_ms, 1.0)
            if switched_cost >= same_mode_cost:
                decode_mode = prefill_mode
                switch_overhead_ms = 0.0
        return SelectionDecision(self.name, prefill_mode, decode_mode, switch_overhead_ms)


def oracle_request_decision(
    signals: RequestSignals,
    db: BenchmarkDB,
    quality_floor: float = 0.95,
    switch_penalty_ms: float = 12.0,
) -> SelectionDecision:
    key = workload_key_from_signals(signals)
    best_pair = (SelectorMode.FP16_BASELINE, SelectorMode.FP16_BASELINE)
    best_energy = float("inf")
    for prefill_mode in db.available_modes("prefill", key):
        for decode_mode in db.available_modes("decode", key):
            prefill = db.get(prefill_mode, "prefill", key)
            decode = db.get(decode_mode, "decode", key)
            if prefill is None or decode is None:
                continue
            quality = min(prefill.quality_score, decode.quality_score)
            if quality < quality_floor:
                continue
            if prefill.phase_latency_ms > signals.ttft_target_ms:
                continue
            if decode.phase_latency_ms > signals.tbt_target_ms:
                continue
            energy = (
                signals.prompt_length * prefill.energy_per_token_j
                + signals.predicted_output_length * decode.energy_per_token_j
            )
            if prefill_mode != decode_mode:
                energy += switch_penalty_ms / 1000.0
            if energy < best_energy:
                best_energy = energy
                best_pair = (prefill_mode, decode_mode)
    return SelectionDecision("oracle", best_pair[0], best_pair[1], switch_penalty_ms if best_pair[0] != best_pair[1] else 0.0)


def evaluate_selectors(selectors: List[BaseSelector], db: BenchmarkDB, n_requests: int = 250, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    state = {
        selector.name: {
            "request_latency_ms": [],
            "request_energy_j": [],
            "quality_score": [],
            "prefill_modes": [],
            "decode_modes": [],
            "switches": 0,
            "slo_hits": 0,
        }
        for selector in selectors
    }
    for _ in range(n_requests):
        signals = RequestSignals(
            prompt_length=int(rng.choice([64, 128, 256, 512, 1024, 2048])),
            predicted_output_length=int(rng.choice([32, 64, 128, 256, 512])),
            memory_pressure=float(rng.uniform(0.25, 0.92)),
            ttft_target_ms=float(rng.uniform(35.0, 160.0)),
            tbt_target_ms=float(rng.uniform(8.0, 30.0)),
        )
        key = workload_key_from_signals(signals)
        for selector in selectors:
            decision = selector.select_request(signals, db)
            prefill = db.get(decision.prefill_mode, "prefill", key)
            decode = db.get(decision.decode_mode, "decode", key)
            if prefill is None or decode is None:
                continue
            request_latency_ms = prefill.phase_latency_ms + signals.predicted_output_length * decode.phase_latency_ms + decision.switch_overhead_ms
            request_energy_j = signals.prompt_length * prefill.energy_per_token_j + signals.predicted_output_length * decode.energy_per_token_j
            quality = min(prefill.quality_score, decode.quality_score)
            slo_hit = prefill.phase_latency_ms <= signals.ttft_target_ms and decode.phase_latency_ms <= signals.tbt_target_ms
            row = state[selector.name]
            row["request_latency_ms"].append(request_latency_ms)
            row["request_energy_j"].append(request_energy_j)
            row["quality_score"].append(quality)
            row["prefill_modes"].append(decision.prefill_mode.value)
            row["decode_modes"].append(decision.decode_mode.value)
            row["switches"] += int(decision.prefill_mode != decision.decode_mode)
            row["slo_hits"] += int(slo_hit)
    rows = []
    for selector in selectors:
        row = state[selector.name]
        n = len(row["request_latency_ms"])
        if n == 0:
            continue
        rows.append(
            {
                "selector": selector.name,
                "total_energy_j": float(np.sum(row["request_energy_j"])),
                "avg_request_latency_ms": float(np.mean(row["request_latency_ms"])),
                "p95_request_latency_ms": float(np.percentile(row["request_latency_ms"], 95)),
                "avg_quality": float(np.mean(row["quality_score"])),
                "slo_hit_rate": row["slo_hits"] / n,
                "unique_prefill_modes": len(set(row["prefill_modes"])),
                "unique_decode_modes": len(set(row["decode_modes"])),
                "switch_rate": row["switches"] / n,
            }
        )
    return pd.DataFrame(rows).sort_values(["total_energy_j", "avg_request_latency_ms"])


def build_default_selectors() -> List[BaseSelector]:
    selectors: List[BaseSelector] = [StaticSelector(mode) for mode in SelectorMode]
    selectors.append(PhaseAgnosticSelector())
    selectors.append(PhaseAwareRuleSelector(allow_switching=False))
    selectors.append(PhaseAwareRuleSelector(allow_switching=True))
    return selectors


def _resolve_mode_name(raw_value: object) -> Optional[SelectorMode]:
    if raw_value is None:
        return None
    text = str(raw_value).strip().lower()
    for mode in SelectorMode:
        if text == mode.value:
            return mode
    return None


def _first_number(row: Mapping[str, object], *keys: str) -> Optional[float]:
    for key in keys:
        if key not in row:
            continue
        value = row[key]
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except Exception:
            pass
        try:
            return float(value)
        except (TypeError, ValueError):
            continue
    return None


def _workload_key_from_row(row: Mapping[str, object]) -> Optional[WorkloadKey]:
    workload_name = row.get("workload_name")
    if workload_name is not None:
        return _workload_key_from_name(str(workload_name))
    workload_cell = row.get("workload_cell")
    if workload_cell is not None:
        mapping = {
            "SS": WorkloadKey("short_prompt", "short_output", "baseline"),
            "SL": WorkloadKey("short_prompt", "long_output", "baseline"),
            "LS": WorkloadKey("long_prompt", "short_output", "baseline"),
            "LL": WorkloadKey("long_prompt", "long_output", "baseline"),
        }
        return mapping.get(str(workload_cell).strip().upper())
    return None


def _workload_key_from_name(workload_name: str) -> Optional[WorkloadKey]:
    name = workload_name.strip().lower()
    if name == "short_prompt_short_output":
        return WorkloadKey("short_prompt", "short_output", "baseline")
    if name == "short_prompt_long_output":
        return WorkloadKey("short_prompt", "long_output", "baseline")
    if name == "long_prompt_short_output":
        return WorkloadKey("long_prompt", "short_output", "baseline")
    if name == "long_prompt_long_output":
        return WorkloadKey("long_prompt", "long_output", "baseline")
    if name.startswith("shared_prefix_chat"):
        return WorkloadKey("long_prompt", "medium_output", "baseline")
    if name == "memory_pressure_long_context":
        return WorkloadKey("long_prompt", "long_output", "mem_pressure_75")
    return None


## Current Repo Alignment

This notebook is self-contained so you only need one notebook file for your deliverable.

It is focused on:

1. showing the current selector assumptions
2. evaluating static, phase-agnostic, and phase-aware baselines
3. keeping a clean hook for Aman/Ali's real benchmark exports later


In [ ]:
print("Selector modes aligned with the repo:")
for mode in SelectorMode:
    print(" -", mode.value)


In [ ]:
aman_additions = pd.DataFrame(
    [
        {
            "artifact": "baseline_smoke_test (3) (1).ipynb",
            "why_it_matters": "controller-style benchmark sweep across the current mode set",
            "selector_takeaway": "compare phase-aware selector against static and phase-agnostic baselines on the same mode vocabulary",
        },
        {
            "artifact": "stress_test (2).ipynb",
            "why_it_matters": "adds long-context and longer-output pressure cases",
            "selector_takeaway": "prompt length, output length, and memory pressure should meaningfully affect routing",
        },
    ]
)
aman_additions


## Benchmark Database

For now we use the synthetic fallback built directly into this notebook, and it is intentionally shaped
to reflect the repo’s current benchmarked modes:

- `prefix_caching` and `chunked_prefill` can help prefill-heavy cases
- `speculative_decoding` can help decode-heavy cases
- `kv_cache_compression` helps more under memory pressure / longer decode
- quantized modes trade off quality for energy / latency / memory


In [ ]:
db = build_synthetic_selector_db(seed=42)
db_df = db.to_dataframe()
print("Rows in selector benchmark DB:", len(db_df))
db_df.head(12)


## Example Runtime Signals

These are the runtime features your part of the project is supposed to use:

- prompt length
- predicted output length
- memory pressure
- TTFT target
- TBT target


In [ ]:
example_signals = RequestSignals(
    prompt_length=1024,
    predicted_output_length=256,
    memory_pressure=0.82,
    ttft_target_ms=140.0,
    tbt_target_ms=12.0,
)

print(example_signals)
print("Mapped workload key:", workload_key_from_signals(example_signals))


## Oracle Reference

The oracle is an upper bound for evaluation. It has full knowledge of the benchmark DB and picks the
lowest-energy valid prefill/decode pair that still satisfies quality and latency constraints.


In [ ]:
oracle_decision = oracle_request_decision(example_signals, db)
oracle_decision


## Selectors

We evaluate:

- static baselines: always one mode
- phase-agnostic baseline: one mode per full request, but still signal-aware
- phase-aware single-mode: phase-aware scoring but no cross-phase switch
- phase-aware switching: can choose different prefill and decode modes when the gain justifies the overhead

This is the comparison structure your proposal calls for.


In [ ]:
selectors = build_default_selectors()
results_df = evaluate_selectors(selectors, db, n_requests=250, seed=7)
results_df


## Focus on the Baselines Required for Hivansh's Part

These are the key rows for the final project narrative:

- `phase_agnostic`
- `phase_aware_rule_single_mode`
- `phase_aware_rule_switch`


In [ ]:
results_df[
    results_df["selector"].isin(
        ["phase_agnostic", "phase_aware_rule_single_mode", "phase_aware_rule_switch"]
    )
].sort_values(["total_energy_j", "avg_request_latency_ms"])


## Quick Read of the Logic

What the updated selector is doing:

- prefill scoring emphasizes TTFT and long-prompt behavior
- decode scoring emphasizes TBT and predicted output length
- memory-heavy modes get penalized when memory pressure is high
- low-quality modes get penalized below a quality floor
- phase-aware switching is only kept if it still beats staying on one mode after adding switch overhead


In [ ]:
phase_agnostic = PhaseAgnosticSelector()
phase_aware_single = PhaseAwareRuleSelector(allow_switching=False)
phase_aware_switch = PhaseAwareRuleSelector(allow_switching=True)

test_cases = [
    RequestSignals(128, 64, 0.35, 60.0, 20.0),
    RequestSignals(1024, 64, 0.35, 140.0, 20.0),
    RequestSignals(1024, 256, 0.85, 150.0, 10.0),
    RequestSignals(2048, 512, 0.88, 180.0, 9.0),
]

rows = []
for signals in test_cases:
    rows.append(
        {
            "signals": repr(signals),
            "phase_agnostic": phase_agnostic.select_request(signals, db),
            "phase_aware_single": phase_aware_single.select_request(signals, db),
            "phase_aware_switch": phase_aware_switch.select_request(signals, db),
        }
    )

pd.DataFrame(rows)


## Where This Connects to Aman’s Work

From the current repo state and Aman’s notebooks:

- `config.py` defines the mode set this selector should reason over
- `baseline_smoke_test (3) (1).ipynb` shows controller-style benchmarking across those modes
- `stress_test (2).ipynb` shows more long-context pressure cases that matter for the selector

So the next integration step is not rewriting the selector again. It is loading real benchmark rows
into `BenchmarkDB.from_result_rows(...)` once the agreed benchmark export is available.


In [ ]:
repo_files = [
    "config.py",
    "modes.py",
    "runner.py",
    "baseline_smoke_test (3) (1).ipynb",
    "stress_test (2).ipynb",
]

for path in repo_files:
    print(Path(path).resolve())


## What Is Complete vs What Still Depends on Teammates

Complete in this notebook:

- selector framing matched to the repo
- static baselines
- phase-agnostic baseline
- phase-aware selector logic
- request-level evaluation
- hook for real benchmark ingestion

Still dependent on teammate outputs:

- final benchmark export from Aman/Ali
- final agreed benchmark schema for loading rows directly
- any practical constraint on whether true cross-phase switching is allowed at runtime

Planned flow:

1. export Aman/Ali benchmark rows to CSV/JSON
2. load them into `BenchmarkDB.from_result_rows(...)`
3. rerun the evaluation cells here
4. report:
   - comparison vs static baselines
   - comparison vs phase-agnostic baseline
   - where switching helps or does not help


In [ ]:
# Example stub for later, once real rows are available:
#
# real_rows = pd.read_csv("path_to_real_benchmark_export.csv").to_dict(orient="records")
# real_db = BenchmarkDB.from_result_rows(real_rows)
# real_results = evaluate_selectors(build_default_selectors(), real_db)
# real_results


## Bottom Line

This notebook now matches your part of the proposal more closely:

- selector design: yes
- phase-agnostic baseline: yes
- static baselines: yes
- runtime signals from the proposal: yes
- phase-aware logic tied to Aman’s updated mode set: yes
- request-level evaluation: yes
- real benchmark integration hook: yes

The remaining dependency is the final benchmark export, not the selector structure itself.
